# Работа от best-модели (LB 0.5539)

Якорь: **`output/cross_tta_gray_a015/soup_run`** — cross-soup `(1−α)·v2@2000 + α·v3@400`, **α=0.15**, symmetry TTA, pair_text **v1**.

| | |
|---|---|
| LB | **0.5539** |
| offline gray (TTA) | 0.5544 |
| offline problem | 0.6357 |

Конфиг: `output/BASE_MODEL.json`

## Дальше (по приоритету)
1. **Fine α** 0.16–0.18 + TTA (~15 мин) — этот ноутбук, cell 3
2. **LB submit** — если α лучше, `build_submit.py` (cell 4)
3. **Fashion hardmine** — teacher = `export_fp16` best, 0.5ep от v2@2000, pair_text v1
4. **Inference rules** — только Обувь/Одежда поверх zip

In [ ]:
import json
from pathlib import Path

START = Path.cwd().resolve()
BASE_JSON = START / "output" / "BASE_MODEL.json"
BASE = json.loads(BASE_JSON.read_text())
RUN_DIR = (START / BASE["paths"]["run_dir"].split("symmetry_tta_v3_soup/")[-1]).resolve()
if not RUN_DIR.exists():
    RUN_DIR = START / "output" / "cross_tta_gray_a015" / "soup_run"

metrics = json.loads((RUN_DIR / "metrics.json").read_text())
print("BASE:", BASE["name"], "| LB", BASE["lb_macro_ap"])
print("run_dir:", RUN_DIR)
print("best_metrics:", json.dumps(metrics["best_metrics"], indent=2))

In [ ]:
import os, subprocess, sys
from pathlib import Path

import pandas as pd

def resolve_script(rel: str) -> Path:
    for base in [START, *START.parents]:
        cand = (base / rel).resolve()
        if cand.exists():
            return cand
    raise FileNotFoundError(rel)

PROJECT = resolve_script("final_4_models/scripts/blend_checkpoint_soup.py").parent.parent.parent
BLEND = PROJECT / "final_4_models/scripts/blend_checkpoint_soup.py"
LIB = PROJECT / "notebooks/lib"
sys.path.insert(0, str(LIB))
from verify_pair_text import patch_score_ensemble_v1
patch_score_ensemble_v1()

CKPT_A = Path(BASE["paths"]["v3_ckpt"])
CKPT_B = Path(BASE["paths"]["v2_ckpt"])
if not CKPT_A.is_absolute():
    CKPT_A = PROJECT / CKPT_A
if not CKPT_B.is_absolute():
    CKPT_B = PROJECT / CKPT_B

FINE_OUT = START / "output" / "cross_tta_fine_alpha" / "soup_run"
GPU_ID = os.getenv("TTA_GPU", "1")
FINE_ALPHAS = "0.16,0.17,0.18"
MIN_PROBLEM_AP = 0.632

cmd = (
    f"cd {PROJECT} && PYTHONPATH={BLEND.parent}:{LIB} python {BLEND} "
    f"--ckpt-a {CKPT_A} --ckpt-b {CKPT_B} --out-dir {FINE_OUT} "
    f"--alphas {FINE_ALPHAS} --skip-submit --gpu {GPU_ID} --symmetry-tta"
)
print("$", cmd.replace(str(PROJECT), "***"))
assert subprocess.run(cmd, shell=True).returncode == 0

grid = pd.DataFrame(json.loads((FINE_OUT / "soup_grid.json").read_text()))
display(grid.round(4))
ok = grid[grid.problem_ap >= MIN_PROBLEM_AP]
pick = ok.loc[ok.gray_full.idxmax()] if len(ok) else grid.loc[grid.gray_full.idxmax()]
print(f"\nPareto pick (gray max, problem≥{MIN_PROBLEM_AP}): α={pick.alpha_step400} gray={pick.gray_full:.4f} problem={pick.problem_ap:.4f}")

In [ ]:
# Pack submit for chosen α (set SUBMIT_ALPHA manually or from fine grid)
import subprocess

BUILD = START / "build_submit.py"
SUBMIT_ALPHA = 0.18          # change after fine grid
SUBMIT_RUN = START / "output" / f"cross_tta_a{int(SUBMIT_ALPHA*100):03d}" / "soup_run"

# re-export single α if needed
if not (SUBMIT_RUN / "export_fp16").exists():
    SUBMIT_RUN.parent.mkdir(parents=True, exist_ok=True)
    cmd = (
        f"cd {PROJECT} && PYTHONPATH={BLEND.parent}:{LIB} python {BLEND} "
        f"--ckpt-a {CKPT_A} --ckpt-b {CKPT_B} --out-dir {SUBMIT_RUN} "
        f"--alphas {SUBMIT_ALPHA} --skip-submit --gpu {GPU_ID} --symmetry-tta"
    )
    assert subprocess.run(cmd, shell=True).returncode == 0

assert subprocess.run(["python", str(BUILD), "--run-dir", str(SUBMIT_RUN)]).returncode == 0
zip_path = SUBMIT_RUN / "matching-bge-human-ft-submit.zip"
print("submit:", zip_path, f"{zip_path.stat().st_size/1024**2:.1f} MB")

## Следующий эксперимент: fashion hardmine

Teacher scorer: `cross_tta_gray_a015/soup_run/export_fp16`  
Init: v2@2000, pair_text **v1**, 0.5 epoch, replay fashion-heavy.

Не использовать v1.5 — ломает gray holdout.